# Notebook 16: LLM Layer with Evaluation

**Question:** Can an AI model turn our CPAP findings into plain-language summaries without inventing facts, and can we prove it?

**Input:** `cpap_resupply_share_v` only (no identifiers, groups of 11+ prescribers).

**Evaluation:** Every number in every AI summary is checked against the source data. Any number not found in the input counts as a fabrication.

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
endpoints = [e.name for e in w.serving_endpoints.list()]
llms = [e for e in endpoints if any(k in e.lower() for k in ["llama", "claude", "gpt", "gemma", "qwen", "mixtral", "dbrx"])]
print(f"{len(endpoints)} endpoints available. Language models:")
for e in llms:
    print(" -", e)

In [0]:
from pyspark.sql import functions as F

ANNUAL_MAX = {"A7030": 4, "A7031": 12, "A7032": 24, "A7033": 24, "A7034": 4, "A7035": 2,
              "A7036": 2, "A7037": 4, "A7038": 24, "A7039": 2, "A7046": 2, "A4604": 4}
DESC = {"A4604": "heated tubing", "A7030": "full face mask", "A7031": "full face mask cushion",
        "A7032": "nasal cushion", "A7033": "nasal pillows", "A7034": "nasal mask",
        "A7035": "headgear", "A7036": "chinstrap", "A7037": "standard tubing",
        "A7038": "disposable filter", "A7039": "non-disposable filter", "A7046": "humidifier water chamber"}

v = spark.table("workspace.default.cpap_resupply_share_v")
items = (v.groupBy("HCPCS_CD")
         .agg(F.sum("prescribers").alias("prescribers"),
              F.round(F.sum(F.col("median_per_patient") * F.col("prescribers")) / F.sum("prescribers"), 1).alias("typical_per_patient"),
              F.sum("robust_high_flags").alias("high_flags"),
              F.sum("robust_low_flags").alias("low_flags"))
         .toPandas())
items["item"] = items.HCPCS_CD.map(DESC)
items["annual_max"] = items.HCPCS_CD.map(ANNUAL_MAX)
items = items.dropna(subset=["item"]).sort_values("HCPCS_CD").reset_index(drop=True)

print(f"High flags total: {items.high_flags.sum()} | Low flags total: {items.low_flags.sum()}")
display(items)

In [0]:
MODEL = "databricks-meta-llama-3-3-70b-instruct"

sdf = spark.createDataFrame(items)
sdf = sdf.withColumn("prompt", F.format_string(
    "You write for a healthcare operations executive. Using ONLY the facts below, write exactly two sentences "
    "about Medicare CPAP resupply for this item. Do not calculate new numbers or percentages, and do not add any "
    "number that is not listed. Facts: item = %s; prescribers = %d; typical units per patient per year = %.1f; "
    "Medicare annual limit per patient = %d; prescribers far above peers = %d; prescribers far below peers = %d.",
    "item", "prescribers", "typical_per_patient", "annual_max", "high_flags", "low_flags"))

out = sdf.withColumn("summary", F.expr(f"ai_query('{MODEL}', prompt)"))
out.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.cpap_llm_summaries")
display(spark.table("workspace.default.cpap_llm_summaries").select("HCPCS_CD", "item", "summary"))

In [0]:
import re
import pandas as pd

rows = spark.table("workspace.default.cpap_llm_summaries").toPandas()

def numbers_in(text):
    return {float(x.replace(",", "")) for x in re.findall(r"\d[\d,]*\.?\d*", text)}

results = []
for _, r in rows.iterrows():
    allowed = {float(r.prescribers), float(r.typical_per_patient), float(r.annual_max),
               float(r.high_flags), float(r.low_flags)}
    found = numbers_in(r.summary)
    unsupported = sorted(found - allowed)
    results.append({"HCPCS_CD": r.HCPCS_CD, "item": r["item"], "numbers_found": len(found),
                    "unsupported": str(unsupported), "passed": len(unsupported) == 0})

res = pd.DataFrame(results)
print(f"Summaries with unsupported numbers: {(~res['passed']).sum()} of {len(res)}")
display(res)

In [0]:
JUDGE = "databricks-qwen3-next-80b-a3b-instruct"

judged = (spark.table("workspace.default.cpap_llm_summaries")
    .withColumn("judge_prompt", F.concat(
        F.lit("You are an auditor. Compare the SUMMARY to the FACTS. Flag any statement the facts do not support, "
              "including claims about what patients need, judgments about prescriber behavior, or a wider scope "
              "than the facts describe. Reply 'SUPPORTED' if there are none; otherwise reply 'ISSUE: ' and the problem in one sentence."
              "\nFACTS: "), F.col("prompt"), F.lit("\nSUMMARY: "), F.col("summary")))
    .withColumn("verdict", F.expr(f"ai_query('{JUDGE}', judge_prompt)")))

judged.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.cpap_llm_judged")
display(spark.table("workspace.default.cpap_llm_judged").select("HCPCS_CD", "item", "verdict"))

## Findings: Notebook 16

**Setup:** An AI model (Llama 3.3 70B, hosted in Databricks) wrote two-sentence summaries for 12 CPAP supply items, using only the shareable view (no identifiers). It was instructed to use only the given facts and not to calculate new numbers.

**Layer 1 (code check on numbers):** Every number was extracted from every summary and matched against the facts. **0 of 12 summaries contained an unsupported number** (51 numbers verified).

**Layer 2 (a second AI model as auditor, Qwen3 80B):** Marked all 12 summaries SUPPORTED.

**Human review found what both layers missed:**
- "Patients *require* 2.1 units": the data measures what patients received, not what they need.
- "Prescribers of CPAP resupply items *like* disposable filters": wider scope than the facts.
- "Prescribing *habits*": frames a statistical signal as behavior.
- Checker gap: numbers written as words ("two," "none") are not checked.

**What I'd tell a COO:** The AI writes the summary and code verifies every digit, but wording, scope, and framing still need review by someone who knows what the data does and doesn't measure. Numeric accuracy is not the same as truthfulness.

**What this test can't prove:** This is one prompt, one judge, and 12 summaries. It shows what patients received, not what they could have had or should have had.

**What I'd need in production:** Calculate derived figures (such as the share of the allowance received) in code first and pass them in as verified facts; check written-out numbers; use a controlled vocabulary ("received," not "require"; "signal," not "habits"); and keep a domain-expert review step before AI summaries reach executives.